In [3]:
import cv2
import numpy as np
import os

# Load COCO class labels
with open('models/coco_class_labels.txt', 'r') as f:
    class_names = f.read().strip().split('\n')
if class_names[0].lower() != 'background':
    class_names.insert(0, 'background')

# Allowed object labels for display
allowed_objects = {'cell phone', 'cup', 'bottle', 'pen', 'remote', 'wallet', 'book'}

# Load DNN object detection model
net = cv2.dnn.readNetFromTensorflow(
    'models/frozen_inference_graph.pb',
    'models/ssd_mobilenet_v2_coco_2018_03_29.pbtxt'
)

# Load face recognizer (LBPH)
face_recognizer = cv2.face.LBPHFaceRecognizer_create()

# Load and train face images
def load_faces_for_training(path='assets/images/'):
    images, labels, label_names = [], [], {}
    name_to_id = {}
    current_id = 0

    for file in os.listdir(path):
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(path, file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"[WARNING] Skipping unreadable image: {img_path}")
                continue
            name = os.path.splitext(file)[0].split('_')[0]
            if name not in name_to_id:
                name_to_id[name] = current_id
                label_names[current_id] = name
                current_id += 1
            label_id = name_to_id[name]
            images.append(cv2.resize(img, (200, 200)))
            labels.append(label_id)
            print(f"[LOADED] {file} as {name} (label ID: {label_id})")

    print("[INFO] Final label map:", label_names)
    return images, np.array(labels), label_names

# Train the recognizer
train_imgs, train_labels, label_map = load_faces_for_training()
face_recognizer.train(train_imgs, train_labels)

# Load face detection model
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Start webcam
cap = cv2.VideoCapture(0)
print("[INFO] Webcam started. Press 'q' to exit.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("[ERROR] Frame capture failed.")
        break

    h, w = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Face detection and recognition
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    for (x, y, wf, hf) in faces:
        face = gray[y:y+hf, x:x+wf]
        face_resized = cv2.resize(face, (200, 200))
        label_id, confidence = face_recognizer.predict(face_resized)
        name = label_map[label_id] if confidence < 120 else "Unknown"
        cv2.rectangle(frame, (x, y), (x+wf, y+hf), (255, 0, 0), 2)
        cv2.putText(frame, f"{name} ({round(confidence, 1)})", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    # Object detection using DNN
    blob = cv2.dnn.blobFromImage(frame, size=(300, 300), swapRB=True, crop=False)
    net.setInput(blob)
    detections = net.forward()

    for i in range(detections.shape[2]):
        conf = detections[0, 0, i, 2]
        if conf > 0.4:
            class_id = int(detections[0, 0, i, 1])
            label = class_names[class_id] if class_id < len(class_names) else "Unknown"
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            x1, y1, x2, y2 = box.astype(int)
            area = (x2 - x1) * (y2 - y1)

            if area > 2000 and label in allowed_objects:
                print(f"[OBJECT] {label} ({conf:.2f}) at [{x1},{y1},{x2},{y2}]")
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("Face + Object Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


[LOADED] Pradyumna_1.jpg as Pradyumna (label ID: 0)
[LOADED] Pradyumna_2.jpg as Pradyumna (label ID: 0)
[LOADED] Vyshaak_1.jpg as Vyshaak (label ID: 1)
[LOADED] Vyshaak_2.jpg as Vyshaak (label ID: 1)
[LOADED] Vyshaak_3.jpg as Vyshaak (label ID: 1)
[INFO] Final label map: {0: 'Pradyumna', 1: 'Vyshaak'}
[INFO] Webcam started. Press 'q' to exit.


In [2]:
import cv2
import numpy as np
import os

# Parameters
FACE_CONFIDENCE_THRESHOLD = 90
ALLOWED_OBJECTS = {'cell phone', 'cup', 'bottle', 'pen', 'remote', 'wallet', 'book'}

# Load COCO class labels
with open('models/coco_class_labels.txt', 'r') as f:
    class_names = f.read().strip().split('\n')
if class_names[0].lower() != 'background':
    class_names.insert(0, 'background')

# Load DNN model
net = cv2.dnn.readNetFromTensorflow(
    'models/frozen_inference_graph.pb',
    'models/ssd_mobilenet_v2_coco_2018_03_29.pbtxt'
)

# Load face recognizer (LBPH)
face_recognizer = cv2.face.LBPHFaceRecognizer_create()

def load_faces(path='assets/images/'):
    images, labels, label_names = [], [], {}
    name_to_id = {}
    current_id = 0

    for file in os.listdir(path):
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(path, file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"[WARNING] Skipping unreadable image: {img_path}")
                continue

            name = os.path.splitext(file)[0].split('_')[0]  # Get name before underscore
            if name not in name_to_id:
                name_to_id[name] = current_id
                label_names[current_id] = name
                current_id += 1

            label_id = name_to_id[name]
            resized_img = cv2.resize(img, (200, 200))
            images.append(resized_img)
            labels.append(label_id)
            print(f"[TRAINING] Loaded {file} as {name} (Label {label_id})")

    print("[INFO] Face Labels Map:", label_names)
    return images, np.array(labels), label_names

# Load and train face recognizer
train_imgs, train_labels, label_map = load_faces()
face_recognizer.train(train_imgs, train_labels)

# Load Haar cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Start webcam
cap = cv2.VideoCapture(0)
print("[INFO] Webcam started. Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("[ERROR] Failed to read frame.")
        break

    h, w = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    for (x, y, wf, hf) in faces:
        face_roi = gray[y:y+hf, x:x+wf]
        face_resized = cv2.resize(face_roi, (200, 200))

        label_id, confidence = face_recognizer.predict(face_resized)
        name = label_map[label_id] if confidence < FACE_CONFIDENCE_THRESHOLD else "Unknown"

        print(f"[FACE DETECTED] {name} | Confidence: {round(confidence, 2)}")
        cv2.rectangle(frame, (x, y), (x+wf, y+hf), (255, 0, 0), 2)
        cv2.putText(frame, f"{name} ({round(confidence, 1)})", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    # Detect objects
    blob = cv2.dnn.blobFromImage(frame, size=(300, 300), swapRB=True, crop=False)
    net.setInput(blob)
    detections = net.forward()

    for i in range(detections.shape[2]):
        conf = detections[0, 0, i, 2]
        if conf > 0.4:
            class_id = int(detections[0, 0, i, 1])
            label = class_names[class_id] if class_id < len(class_names) else "Unknown"
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            x1, y1, x2, y2 = box.astype(int)

            area = (x2 - x1) * (y2 - y1)
            if area > 2000 and label in ALLOWED_OBJECTS:
                print(f"[OBJECT DETECTED] {label} ({round(conf * 100, 1)}%)")
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("Live Face + Object Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


[TRAINING] Loaded Pradyumna_1.jpg as Pradyumna (Label 0)
[TRAINING] Loaded Pradyumna_2.jpg as Pradyumna (Label 0)
[TRAINING] Loaded Vyshaak_1.jpg as Vyshaak (Label 1)
[TRAINING] Loaded Vyshaak_2.jpg as Vyshaak (Label 1)
[TRAINING] Loaded Vyshaak_3.jpg as Vyshaak (Label 1)
[INFO] Face Labels Map: {0: 'Pradyumna', 1: 'Vyshaak'}
[INFO] Webcam started. Press 'q' to quit.
[FACE DETECTED] Unknown | Confidence: 102.85
[FACE DETECTED] Unknown | Confidence: 103.45
[FACE DETECTED] Unknown | Confidence: 103.54
[FACE DETECTED] Unknown | Confidence: 103.57
[FACE DETECTED] Unknown | Confidence: 102.47
[FACE DETECTED] Unknown | Confidence: 103.54
[FACE DETECTED] Unknown | Confidence: 104.78
[FACE DETECTED] Unknown | Confidence: 103.24
[FACE DETECTED] Unknown | Confidence: 102.02
[FACE DETECTED] Unknown | Confidence: 101.5
[FACE DETECTED] Unknown | Confidence: 101.47
[FACE DETECTED] Unknown | Confidence: 101.48
[FACE DETECTED] Unknown | Confidence: 101.12
[FACE DETECTED] Unknown | Confidence: 102.31
[